# M8 · Information Theory — companion notebook

> **Play with this.** A *demonstration, not an assessment* — the module's real assessment is its problem set. Everything here makes bits physical: measure the entropy of real text, build a working Huffman code and watch its average length hug the entropy from above, pay the cross-entropy bill for using the wrong model, and convert a language model's loss into perplexity.

Companion to the **Information Theory** module — the final module of the Mathematical Foundations track at [llmsforsocialscience.net](https://llmsforsocialscience.net/).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

rng = np.random.default_rng(8)

def entropy_bits(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

## 1 · The entropy of real text

Measure expected surprise at the character level on a small on-brand corpus. Uniform over 27 symbols (26 letters + space) would cost log₂ 27 ≈ 4.75 bits per character; real English frequencies cost far less — and that gap is structure a model can learn.

In [ ]:
text = (
    "the survey results were coded by three independent annotators and the model "
    "was validated against the human labels before any downstream analysis was run "
    "the researchers reported agreement metrics alongside the raw accuracy because "
    "a classifier is a measurement instrument and measurement instruments must be defended"
)

chars = [c for c in text.lower() if c.isalpha() or c == " "]
freq = Counter(chars)
p_chars = np.array([freq[c] for c in sorted(freq)]) / len(chars)

H_char = entropy_bits(p_chars)
print(f"characters in corpus:        {len(chars)}")
print(f"distinct symbols:            {len(freq)}")
print(f"uniform cost:                {np.log2(len(freq)):.2f} bits/char")
print(f"unigram entropy:             {H_char:.2f} bits/char")
print(f"structure already saved:     {(1 - H_char/np.log2(len(freq)))*100:.0f}%")
print("\n(and this counts only letter frequencies — digram, word, and sentence")
print(" structure keep lowering it; Shannon put well-modelled English near 1 bit/char)")

## 2 · A working Huffman code

The module's widget, rebuilt in fifteen lines: common symbols get short codewords, rare ones long, and the average length lands just above the entropy — never below. This is the source coding theorem happening on your screen.

In [ ]:
def huffman(p_dict):
    """p_dict: symbol -> probability. Returns symbol -> bitstring."""
    nodes = [(p, i, {s: ""}) for i, (s, p) in enumerate(p_dict.items())]
    counter = len(nodes)
    while len(nodes) > 1:
        nodes.sort(key=lambda t: (t[0], t[1]))
        (pa, _, ca), (pb, _, cb) = nodes.pop(0), nodes.pop(0)
        merged = {s: "0" + c for s, c in ca.items()} | {s: "1" + c for s, c in cb.items()}
        nodes.append((pa + pb, counter, merged)); counter += 1
    return nodes[0][2]

p_dict = {c: freq[c] / len(chars) for c in sorted(freq)}
codes = huffman(p_dict)
avg_len = sum(p_dict[s] * len(codes[s]) for s in p_dict)

print("symbol  p      codeword")
for s in sorted(p_dict, key=p_dict.get, reverse=True)[:8]:
    label = "space" if s == " " else s
    print(f"{label:6s}  {p_dict[s]:.3f}  {codes[s]}")
print("  ...")
print(f"\nentropy:          {H_char:.3f} bits/char   <- the floor")
print(f"Huffman average:  {avg_len:.3f} bits/char   <- just above it")
print(f"fixed-width code: {np.ceil(np.log2(len(p_dict))):.0f} bits/char")

## 3 · The bill for the wrong model

Encode one text with a code built for *another* text's frequencies. The average length rises from H(p) to (about) the cross-entropy H(p, q), and the excess is the KL divergence — the module's §3–§4 in four lines of numpy.

In [ ]:
# A second corpus with different letter statistics (methods-speak vs weather-speak)
other = (
    "heavy rain moved across the region overnight with wind gusts near the coast "
    "and morning fog expected to clear by noon while temperatures stay mild all week"
)
chars_q = [c for c in other.lower() if c.isalpha() or c == " "]
freq_q = Counter(chars_q)

symbols = sorted(set(freq) | set(freq_q))
eps = 1e-9
p = np.array([freq.get(s, 0) for s in symbols], dtype=float); p /= p.sum()
q = np.array([freq_q.get(s, 0) for s in symbols], dtype=float); q = (q + eps) / (q + eps).sum()

Hp = entropy_bits(p)
Hpq = float(-(p[p > 0] * np.log2(q[p > 0])).sum())
print(f"H(p)   (own code):    {Hp:.3f} bits/char")
print(f"H(p,q) (wrong code):  {Hpq:.3f} bits/char")
print(f"KL(p||q) (the bill):  {Hpq - Hp:.3f} bits/char")

# And realised with an actual Huffman code built for q:
codes_q = huffman({s: qi for s, qi in zip(symbols, q)})
avg_wrong = sum(pi * len(codes_q[s]) for s, pi in zip(symbols, p) if pi > 0)
print(f"\nHuffman-for-q on p:   {avg_wrong:.3f} bits/char  (cross-entropy + rounding)")

## 4 · KL asymmetry, seen

D(p‖q) and D(q‖p) are different numbers with different behaviour. The extreme case: if q assigns (near-)zero where p has real mass, D(p‖q) explodes — the wrong-model code simply has no short way to say something that keeps happening.

In [ ]:
def kl_bits(p, q):
    mask = p > 0
    return float((p[mask] * np.log2(p[mask] / q[mask])).sum())

pa = np.array([0.5, 0.5])
qa = np.array([0.75, 0.25])
print(f"module's problem 5:  D(p||q) = {kl_bits(pa, qa):.3f}   D(q||p) = {kl_bits(qa, pa):.3f}   (different)")

# The explosion: q nearly rules out an outcome that p produces 30% of the time
pb = np.array([0.7, 0.3])
qb = np.array([0.999, 0.001])
print(f"near-zero q mass:    D(p||q) = {kl_bits(pb, qb):.2f} bits   D(q||p) = {kl_bits(qb, pb):.2f} bits")
print("\nthe first direction punishes q for failing to cover p — which is why the")
print("policy goes in the FIRST slot of the preference-tuning penalty D(policy||reference):")
print("the policy may narrow, but may not invent behaviour the reference rules out.")

## 5 · Perplexity of an actual language model

A bigram model, trained by counting (M7's estimator), evaluated properly: average cross-entropy on the text, then exponentiated into perplexity — versus the uniform model's vocabulary-sized perplexity.

In [ ]:
words = text.split()
vocab = sorted(set(words))
V = len(vocab)
idx = {w: i for i, w in enumerate(vocab)}

# Bigram counts with add-0.1 smoothing (so no zero ever meets the log)
counts = np.full((V, V), 0.1)
for a, b in zip(words, words[1:]):
    counts[idx[a], idx[b]] += 1
bigram = counts / counts.sum(axis=1, keepdims=True)

ce_bigram = -np.mean([np.log2(bigram[idx[a], idx[b]]) for a, b in zip(words, words[1:])])
ce_uniform = np.log2(V)

print(f"vocabulary size:        {V}")
print(f"uniform model:          {ce_uniform:.2f} bits/token  ->  PPL = {2**ce_uniform:.1f}  (= V, knows nothing)")
print(f"bigram model (train):   {ce_bigram:.2f} bits/token  ->  PPL = {2**ce_bigram:.1f}")
print("\n(evaluated on its own training text, so this flatters the bigram —")
print(" the honest version holds out data, which is exactly the module's PPL=1 leakage warning)")

## 6 · Why cross-tokenizer perplexity comparisons fail

The same text, 'tokenized' two ways — by word and by character. Same string, same information; wildly different per-token perplexities. Only the per-character (shared-unit) costs agree on which is which.

In [ ]:
# Per-word cost from the bigram model, expressed in two currencies
total_bits = ce_bigram * (len(words) - 1)          # total for the text under the bigram model
n_chars = len(" ".join(words))

per_word = total_bits / (len(words) - 1)
per_char = total_bits / n_chars
print(f"same model, same text, same total bits: {total_bits:.0f}")
print(f"  per-word  'perplexity':  {2**per_word:7.1f}   (looks bad)")
print(f"  per-char  'perplexity':  {2**per_char:7.2f}   (looks great)")
print("\nneither number is wrong — they are prices in different currencies.")
print("comparing models that tokenize differently requires converting to a shared")
print("unit first: total bits on the same string, or bits per character.")

The track ends here. Entropy will be waiting on the systems side — same number, measured as a storage floor instead of an expected surprise — and the loss you can now derive is the one every training curve in the deep-learning and LLMs tracks plots.

---

**Next:** the Mathematical Foundations track is complete. Continue to the LLMs track — or meet these bits again in Systems S1, from the other side.